This is a hierarchical Negative Binomial model that predicts monthly complaint counts for each NTA-category combination by combining neighborhood structure, reporting intensity, category baselines, month effects, and category-specific local time dynamics.

In [1]:
import re
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    sys.path.insert(0, str(cwd.parent))
else:
    sys.path.insert(0, str(cwd))


In [2]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pymc as pm
import arviz as az

In [3]:
from helpers import prep_the_data, prep_zone_data, export_idata
from src.models.qol_model import build_reported_qol_pressure_model

In [4]:
df_nta = pd.read_parquet("../data/processed/features/complaints_311_nta_category.parquet")
df_nta = prep_the_data(df_nta)

cutoff = pd.Timestamp("2024-12-31")

prepared_filtered = df_nta[df_nta['complaint_date'] <= cutoff]


# Ensure datetime once (upstream helpers may already do this, but keep it explicit)
prepared_filtered.head()

,complaint_date,time_of_day,nta_id,nta_name,borough,level_1,level_2,complaint_count,category,category_group,dow,month,is_weekend,month_year,dow_complaint
0,2022-01-01,day,BK0101,Greenpoint,Brooklyn,Noise & Disturbance,Commercial Noise,1,Commercial Noise,Noise & Disturbance,Saturday,January,1,January__2022,COMMERCIAL_NOISE__Saturday
1,2022-01-01,day,BK0101,Greenpoint,Brooklyn,Parks & Trees,Tree Requests,1,Tree Requests,Parks & Trees,Saturday,January,1,January__2022,TREE_REQUESTS__Saturday
2,2022-01-01,day,BK0101,Greenpoint,Brooklyn,Sanitation & Waste,Garbage Collection,1,Garbage Collection,Sanitation & Waste,Saturday,January,1,January__2022,GARBAGE_COLLECTION__Saturday
3,2022-01-01,day,BK0101,Greenpoint,Brooklyn,Transportation & Vehicles,Parking Issues,1,Parking Issues,Transportation & Vehicles,Saturday,January,1,January__2022,PARKING_ISSUES__Saturday
4,2022-01-01,day,BK0102,Williamsburg,Brooklyn,Homelessness & Social Services,Encampments,2,Encampments,Homelessness & Social Services,Saturday,January,1,January__2022,ENCAMPMENTS__Saturday


In [5]:
pluto_with_nta, zoning_long, available_zoning_columns = prep_zone_data()

Using zoning columns: ['zonedist1', 'zonedist2']


In [6]:
pluto_features = pd.read_parquet("../data/processed/features/pluto_nta_features.parquet")

df_model = prepared_filtered.copy()
df_model["month_start"] = df_model["complaint_date"].dt.to_period("M").dt.start_time

# Support both legacy (level_1/level_2) and current (category/category_group) schemas.
if {"level_1", "level_2"}.issubset(df_model.columns):
    category_pairs = (
        df_model[["level_1", "level_2"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["level_1", "level_2"])
        .reset_index(drop=True)
    )
    category_pairs["category"] = category_pairs["level_2"]
    if category_pairs["category"].duplicated().any():
        category_pairs["category"] = category_pairs["level_1"] + " / " + category_pairs["level_2"]

    df_model = df_model.drop(columns=["category"], errors="ignore").merge(
        category_pairs,
        on=["level_1", "level_2"],
        how="left",
    )
else:
    if "category" not in df_model.columns:
        raise ValueError(
            "Expected either level_1/level_2 or category columns in df_model."
        )

    if "category_group" not in df_model.columns:
        df_model["category_group"] = "All Categories"

    df_model["level_1"] = df_model["category_group"].astype("string")
    df_model["level_2"] = df_model["category"].astype("string")

    category_pairs = (
        df_model[["level_1", "level_2"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["level_1", "level_2"])
        .reset_index(drop=True)
    )
    category_pairs["category"] = category_pairs["level_2"]
    if category_pairs["category"].duplicated().any():
        category_pairs["category"] = category_pairs["level_1"] + " / " + category_pairs["level_2"]

    df_model = df_model.drop(columns=["category"], errors="ignore").merge(
        category_pairs,
        on=["level_1", "level_2"],
        how="left",
    )

monthly_counts = (
    df_model.groupby(
        ["nta_id", "month_start", "category", "level_1", "level_2"],
        observed=True,
    )["complaint_count"]
    .sum()
    .rename("complaint_count")
)

nta_index = pd.Index(sorted(df_model["nta_id"].dropna().unique()), name="nta_id")
month_index = pd.DatetimeIndex(sorted(df_model["month_start"].dropna().unique()), name="month_start")
category_index = pd.Index(category_pairs["category"], name="category")
category_group_index = pd.Index(sorted(category_pairs["level_1"].unique()), name="category_group")

model_index = pd.MultiIndex.from_frame(
    pd.DataFrame(
        [
            (nta_id, month_start, row.category, row.level_1, row.level_2)
            for nta_id in nta_index
            for month_start in month_index
            for row in category_pairs.itertuples(index=False)
        ],
        columns=["nta_id", "month_start", "category", "level_1", "level_2"],
    )
)

model_df = (
    monthly_counts.reindex(model_index, fill_value=0)
    .reset_index()
)

model_df["nta_idx"] = pd.Categorical(model_df["nta_id"], categories=nta_index).codes.astype("int32")
model_df["month_idx"] = pd.Categorical(model_df["month_start"], categories=month_index).codes.astype("int32")
model_df["cat_idx"] = pd.Categorical(model_df["category"], categories=category_index).codes.astype("int32")
model_df["cat_group_idx"] = pd.Categorical(model_df["level_1"], categories=category_group_index).codes.astype("int32")
model_df["month_of_year"] = model_df["month_start"].dt.month.astype("int16")

zoning_area = (
    zoning_long.groupby(["nta_id", "zoning_group"], observed=True)["allocated_lotarea"]
    .sum()
    .unstack(fill_value=0)
    .reindex(nta_index, fill_value=0)
)
zoning_share = zoning_area.div(zoning_area.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
zoning_share = zoning_share.add_prefix("zoning_share__")

pluto_by_nta = pluto_features.set_index("nta_id").reindex(nta_index)

issue_feature_columns = [
    "res_units_per_acre",
    "bldg_sqft_per_acre",
    "buildings_per_acre",
    "pct_res_area",
    "pct_com_area",
    "pct_mixed_use_lots",
    "pct_prewar",
    "avg_built_far_weighted",
]
reporting_feature_columns = [
    "lot_count",
    "lot_area_acres",
    "units_res_total",
    "units_total",
    "num_buildings_total",
]

issue_features = pd.concat(
    [pluto_by_nta[issue_feature_columns], zoning_share],
    axis=1,
)
reporting_features = pluto_by_nta[reporting_feature_columns].apply(np.log1p)

def standardize_features(features: pd.DataFrame) -> pd.DataFrame:
    numeric = features.apply(pd.to_numeric, errors="coerce")
    std = numeric.std(axis=0, ddof=0).replace(0, 1)
    return ((numeric - numeric.mean(axis=0)) / std).fillna(0)

X_issue_nta = standardize_features(issue_features).to_numpy(dtype="float64")
X_reporting_nta = standardize_features(reporting_features).to_numpy(dtype="float64")

exposure_by_nta = (
    pd.to_numeric(pluto_by_nta["units_res_total"], errors="coerce")
    .fillna(pd.to_numeric(pluto_by_nta["lot_count"], errors="coerce"))
    .fillna(1)
    .clip(lower=1)
)
model_df["exposure"] = model_df["nta_id"].map(exposure_by_nta).astype("float64")

coords = {
    "obs": np.arange(len(model_df)),
    "nta": nta_index.to_list(),
    "month": month_index.strftime("%Y-%m-%d").to_list(),
    "category": category_index.to_list(),
    "category_group": category_group_index.to_list(),
    "issue_feature": issue_features.columns.to_list(),
    "reporting_feature": reporting_features.columns.to_list(),
}



In [7]:
qol_pressure_model = build_reported_qol_pressure_model(
    y=model_df["complaint_count"].to_numpy(dtype="int64"),
    nta_idx=model_df["nta_idx"].to_numpy(),
    month_idx=model_df["month_idx"].to_numpy(),
    cat_idx=model_df["cat_idx"].to_numpy(),
    cat_group_idx=model_df["cat_group_idx"].to_numpy(),
    exposure=model_df["exposure"].to_numpy(),
    X_issue_nta=X_issue_nta,
    X_reporting_nta=X_reporting_nta,
    n_nta=len(nta_index),
    n_month=len(month_index),
    n_cat=len(category_index),
    n_cat_group=len(category_group_index),
    month_of_year=model_df["month_of_year"].to_numpy(),
    coords=coords,
    include_local_state=True,
    include_reporting_structural=False,
)



In [ ]:
with qol_pressure_model:
    idata = pm.sample(
        draws=500,
        tune=500,
        chains=4,
        target_accept=0.85,
        progressbar=True,
        idata_kwargs={"log_likelihood": True}
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta0, beta_issue, beta_reporting, sigma_nta, alpha_nta, sigma_month, gamma_month, sigma_cat, delta_cat, beta_sin, beta_cos, rho, sigma_state, local_state, alpha_nb_cat]


Output()

/Users/mozilla/Library/Caches/pypoetry/virtualenvs/municipal-stress-engine-d7i9H-rK-py3.12/lib/python3.12/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)
/Users/mozilla/Library/Caches/pypoetry/virtualenvs/municipal-stress-engine-d7i9H-rK-py3.12/lib/python3.12/site-packages/pymc/step_methods/hmc/quadpotential.py:316: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


In [ ]:
export_idata(idata, "../data/processed/models/model_urban_pressure_idata-2.nc")

In [ ]:
az.summary(idata)